In [1]:
import numpy as np
import plotly.graph_objects as go
from utils import (
    plotly_draw_3D_mesh,
    plotly_draw_2D_mesh,
    read_obj,
    save_obj,
    generate_torus_mesh_with_normals,
    generate_torus_mesh_with_variable_r,
)
import shapecomp as sc

In [4]:
V, normals, F = generate_torus_mesh_with_normals(20, 30, r=0.9, R=1)
# save_obj(V, F, "data/synthetic/torus_ratio_0.9_n_theta_20_n_phi_10.obj")
fig = plotly_draw_3D_mesh(
    V,
    F,
    show_now=False,
    mesh_color="grey",
)
fig.show()

In [5]:
for r in [0.1, 0.2, 0.5, 0.8, 0.9]:
    V, normals, F = generate_torus_mesh_with_normals(20, 30, r=r, R=1)
    save_obj(
        V,
        F,
        f"data/synthetic/torus_ratio_{r}_n_theta_20_n_phi_30.obj",
    )

In [2]:
for r in [0.1, 0.2, 0.5, 0.8, 0.9]:
    V, normals, F = generate_torus_mesh_with_normals(50, 50, r=r, R=1)
    save_obj(
        V,
        F,
        f"data/synthetic/torus_ratio_{r}_n_theta_50_n_phi_50.obj",
    )

In [ ]:
# geomp supports subdivision by inserting one vertex per edge and one vertex per face
mesh = sc.load_mesh(V, F)  # or sc.load_mesh("path.obj")
sub = sc.subdivide_mesh(mesh)
V_sub = sub.get_vertices()
F_sub = sub.get_faces()

In [ ]:
def barycenter_subdivide_insert_one_vertex_per_face(V, F):
    """
    Barycenter Subdivide the mesh
    For each face
        - Add a vertex at the barycenter of each face, and connect to the three vertices of the face
    """
    # clone the old vertices
    V_old = np.copy(V)
    n_V = V.shape[0]
    V_new = []
    F_subdivided = []

    for f in F:
        # compute the barycenter of the face
        barycenter = np.mean(V[f], axis=0)
        V_new.append(barycenter)
        F_subdivided.append([f[0], f[1], n_V])
        F_subdivided.append([f[1], f[2], n_V])
        F_subdivided.append([f[2], f[0], n_V])
        n_V += 1

    # convert to numpy arrays
    V_new = np.array(V_new)
    F_subdivided = np.array(F_subdivided)
    # concatenate the old vertices and the new vertices
    V_subdivided = np.concatenate([V_old, V_new], axis=0)
    return V_subdivided, F_subdivided


def barycenter_subdivide_insert_one_vertex_per_edge(V, F):
    """
    Barycenter Subdivide the mesh
    For each edge
        - Add a vertex in the middle of the edge, and
        - for each face, connect the 3 new vertices -> replace 1 by 4 faces
    """
    # clone the old vertices
    V_old = np.copy(V)
    n_V = V.shape[0]
    V_new = []
    F_subdivided = []

    # map each edge to the new vertex
    edge_to_new_vertex_map = {}
    for f in F:
        for i in range(3):
            edge = tuple(sorted((f[i], f[(i + 1) % 3])))
            if edge not in edge_to_new_vertex_map:
                edge_to_new_vertex_map[edge] = n_V
                n_V += 1
                V_new.append(np.mean(V[list(edge)], axis=0))

    for f in F:
        # iterate over the 3 edges of the face
        e_1 = tuple(sorted((f[0], f[1])))
        e_2 = tuple(sorted((f[1], f[2])))
        e_3 = tuple(sorted((f[2], f[0])))
        u = edge_to_new_vertex_map[e_1]
        v = edge_to_new_vertex_map[e_2]
        w = edge_to_new_vertex_map[e_3]
        F_subdivided.append([f[0], u, w])
        F_subdivided.append([f[1], v, u])
        F_subdivided.append([f[2], w, v])
        F_subdivided.append([u, v, w])

    # convert to numpy arrays
    V_new = np.array(V_new)
    F_subdivided = np.array(F_subdivided)
    # concatenate the old vertices and the new vertices
    V_subdivided = np.concatenate([V_old, V_new], axis=0)
    return V_subdivided, F_subdivided

In [9]:
V_sub, F_sub = barycenter_subdivide_insert_one_vertex_per_edge(V, F)

In [10]:
fig = plotly_draw_3D_mesh(
    V_sub,
    F_sub,
    show_now=False,
    edge_color_width=3,
    edge_color_opacity=0.4,
    mesh_color_opacity=0.5,
)
fig.show()

In [11]:
for r in [0.1, 0.2, 0.5, 0.8, 0.9]:
    V, normals, F = generate_torus_mesh_with_normals(20, 10, r=r, R=1)
    V_sub, F_sub = barycenter_subdivide_insert_one_vertex_per_edge(V, F)
    save_obj(
        V_sub,
        F_sub,
        f"data/synthetic/torus_ratio_{r}_n_theta_20_n_phi_10_subdivided.obj",
    )

In [11]:
save_obj(V_sub, F_sub, "data/synthetic/b2_subdivide_each_face.obj")

In [3]:
V, normals, F = generate_torus_mesh_with_normals(40, 40, r=0.5, R=1)
save_obj(V, F, "data/synthetic/torus_ratio_0.5_n_theta_40_n_phi_40.obj")

In [3]:
V, normals, F = generate_torus_mesh_with_normals(30, 30, r=0.5, R=1)
save_obj(V, F, "data/synthetic/torus_ratio_0.5_n_theta_30_n_phi_30.obj")
fig = plotly_draw_3D_mesh(
    V,
    F,
    show_now=False,
    edge_color_width=3,
    edge_color_opacity=0.4,
    mesh_color_opacity=0.5,
)
fig.show()

In [3]:
gaussian_bandwidth = 1
gaussian_func = lambda x: np.exp(-(x**2) / gaussian_bandwidth**2)
# V, F = generate_torus_mesh_with_variable_r(20, 10, r=0.5, R=1, func=gaussian_func)
# # save_obj(V, F, "data/synthetic/irregular_gaussain_theta_0.5_20_10.obj")
# fig = plotly_draw_3D_mesh(
#     V,
#     F,
#     show_now=False,
#     edge_color_width=3,
#     edge_color_opacity=0.4,
#     mesh_color_opacity=0.5,
# )
# fig.show()

V, F = generate_torus_mesh_with_variable_r(
    20, 10, r=0.5, R=1, func=gaussian_func, tile_along_theta=False
)
# save_obj(V, F, "data/synthetic/irregular_gaussain_phi_0.5_20_10.obj")
fig = plotly_draw_3D_mesh(
    V,
    F,
    show_now=False,
    edge_color_width=3,
    edge_color_opacity=0.4,
    mesh_color_opacity=0.5,
)
fig.show()

In [4]:
def save_off(vertices: np.ndarray, faces: np.ndarray, filename: str) -> None:
    with open(filename, "w") as file:
        file.write("OFF\n")
        file.write(f"{len(vertices)} {len(faces)} 0\n")
        for v in vertices:
            file.write(f"{v[0]} {v[1]} {v[2]}\n")
        for f in F:
            file.write(f"{len(f)} ")
            # add 3 in the front to denote the number of vertices in the face
            # file.write("3 ")
            for i in f:
                file.write(f"{i} ")
            file.write("\n")


# save_off(V, F, "irregular_gaussain_phi_0.5_20_10.off")

In [21]:
loaded_mesh = sc.load_mesh("data/synthetic/torus_ratio_0.5_n_theta_20_n_phi_10.obj")
V, F = loaded_mesh.get_vertices(), loaded_mesh.get_faces()
path1 = V[
    [
        15,
        35,
        55,
        75,
        95,
        115,
        135,
        155,
        175,
        195,
    ]
]
path2 = V[
    [
        100,
        101,
        102,
        103,
        104,
        105,
        106,
        107,
        108,
        109,
        110,
        111,
        112,
        113,
        114,
        115,
        116,
        117,
        118,
        119,
    ]
]

fig = plotly_draw_3D_mesh(
    V,
    F,
    show_now=False,
    edge_color_width=3,
    mesh_color="grey",
    edge_color_opacity=0.4,
    mesh_color_opacity=0.5,
)

fig.add_trace(
    go.Scatter3d(
        x=path1[:, 0],
        y=path1[:, 1],
        z=path1[:, 2],
        mode="lines",
        line=dict(color="red", width=5),
        name="path1",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=path2[:, 0],
        y=path2[:, 1],
        z=path2[:, 2],
        mode="lines",
        line=dict(color="green", width=5),
        name="path2",
    )
)

fig.show()

In [20]:
path1 = V[[19, 39, 59, 79, 99, 119, 139, 159, 179, 199]]
path2 = V[
    [
        100,
        101,
        102,
        103,
        104,
        105,
        106,
        107,
        108,
        109,
        110,
        111,
        112,
        113,
        114,
        115,
        116,
        117,
        118,
        119,
    ]
]

fig = plotly_draw_3D_mesh(
    V,
    F,
    show_now=False,
    edge_color_width=3,
    mesh_color="grey",
    edge_color_opacity=0.4,
    mesh_color_opacity=0.5,
)

fig.add_trace(
    go.Scatter3d(
        x=path1[:, 0],
        y=path1[:, 1],
        z=path1[:, 2],
        mode="lines",
        line=dict(color="red", width=5),
        name="path1",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=path2[:, 0],
        y=path2[:, 1],
        z=path2[:, 2],
        mode="lines",
        line=dict(color="green", width=5),
        name="path2",
    )
)

fig.show()